# SG-VO: Self-Supervised Geometric Visual Odometry
## Google Colab Evaluation Notebook

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

### What this notebook does:
1. ✅ Checks GPU
2. ✅ Installs all dependencies
3. ✅ Downloads pretrained models from Google Drive
4. ✅ Downloads KITTI odometry sequences 09 & 10 (for evaluation)
5. ✅ Runs offline VO evaluation
6. ✅ Runs online VO evaluation (with adaptation)
7. ✅ Shows results

In [ ]:
# ── CELL 1: Check GPU ─────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('⚠️  NO GPU — Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── CELL 2: Install Dependencies ──────────────────────────────────────────────
!pip install -q pytorch-lightning==1.9.5 kornia einops path configargparse \
    blessings progressbar2 scikit-image imageio gdown
print('✅ Dependencies installed')

In [ ]:
# ── CELL 3: Upload SG-VO Code ─────────────────────────────────────────────────
# Option A: Clone from GitHub (if repo is public)
# !git clone https://github.com/YOUR_REPO/SG-VO.git
# %cd SG-VO

# Option B: Upload zip from local machine
import os
from google.colab import files

if not os.path.exists('/content/SG-VO'):
    print('Upload your SG-VO.zip file when prompted...')
    print('(Create it locally with: cd ~/ICRAMaxxing && zip -r SG-VO.zip SG-VO/ --exclude SG-VO/data/\* SG-VO/.git/\*)')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    !unzip -q "{fname}" -d /content/
    print('✅ SG-VO extracted')
else:
    print('✅ SG-VO already present')

os.chdir('/content/SG-VO')
!ls

In [ ]:
# ── CELL 4: Download Pretrained Models ───────────────────────────────────────
import os
os.makedirs('checkpoints', exist_ok=True)

disp_path  = 'checkpoints/dispnet112_model_best.pth.tar'
pose_path  = 'checkpoints/exp_pose112_model_best.pth.tar'

# Download from Google Drive (models_Dec28 folder)
if not os.path.exists(disp_path):
    print('Downloading pretrained models from Google Drive...')
    !gdown --folder 'https://drive.google.com/drive/folders/1twZsg2pxMLwcUUnFszBn1ryaEtuoEfs3' \
        -O checkpoints/ --quiet
    # Move up from subfolder if needed
    import glob, shutil
    for f in glob.glob('checkpoints/**/*.pth.tar', recursive=True):
        dest = os.path.join('checkpoints', os.path.basename(f))
        if f != dest:
            shutil.move(f, dest)

print('Checkpoints:')
!ls -lh checkpoints/*.pth.tar 2>/dev/null || echo 'Not found — check Drive link'

In [ ]:
# ── CELL 5: Download Camera Intrinsics ───────────────────────────────────────
import os
os.makedirs('data/kitti_odom/sequences/kitti_odom256_intrinsics', exist_ok=True)

if not os.path.exists('data/kitti_odom/sequences/kitti_odom256_intrinsics/cam_09.txt'):
    print('Downloading camera intrinsics from Google Drive...')
    !gdown --folder 'https://drive.google.com/drive/folders/1n81QDHaG3lIxnxybl9I6knPHxPngTsD8' \
        -O data/kitti_odom/sequences/ --quiet

!ls data/kitti_odom/sequences/kitti_odom256_intrinsics/ 2>/dev/null || echo 'Not found'

In [ ]:
# ── CELL 6: Download KITTI Odometry (Sequences 09 & 10 only) ─────────────────
# The full KITTI zip is 65GB. We download only sequences 09 & 10 (~12GB total)
# which are the standard test sequences used in the paper.
import os

os.makedirs('data/kitti_odom/sequences', exist_ok=True)
os.makedirs('vo_results', exist_ok=True)
os.makedirs('vo_results_online', exist_ok=True)

DATASET_DIR = 'data/kitti_odom/sequences/'

# Check available disk space
import shutil
total, used, free = shutil.disk_usage('/')
print(f'Disk: {free//1e9:.0f}GB free of {total//1e9:.0f}GB')

# Download full KITTI odometry zip (Colab has fast internet ~100MB/s → ~10 min)
zip_path = 'data/data_odometry_color.zip'
if not os.path.exists(f'{DATASET_DIR}/09/image_2'):
    if not os.path.exists(zip_path):
        print('Downloading KITTI Odometry Color (~65GB). Colab internet is fast (~10-15 min)...')
        !wget -q --show-progress -c \
            'https://s3.eu-central-1.amazonaws.com/avg-kitti/data_odometry_color.zip' \
            -O {zip_path}
    print('Extracting sequences 09 and 10 only...')
    !unzip -q {zip_path} 'dataset/sequences/09/*' 'dataset/sequences/10/*' -d /tmp/kitti_raw/
    !mv /tmp/kitti_raw/dataset/sequences/09 {DATASET_DIR}
    !mv /tmp/kitti_raw/dataset/sequences/10 {DATASET_DIR}
    !rm -f {zip_path}  # free disk space
    print('✅ Sequences 09 and 10 extracted')
else:
    print('✅ KITTI sequences already present')

!ls {DATASET_DIR}

In [ ]:
# ── CELL 7: Install cam.txt intrinsics for each sequence ─────────────────────
import os, shutil

INTRINSICS_DIR = 'data/kitti_odom/sequences/kitti_odom256_intrinsics'
SEQUENCES_DIR  = 'data/kitti_odom/sequences'

for seq in ['09', '10']:
    cam_src  = f'{INTRINSICS_DIR}/cam_{seq}.txt'
    img_dir  = f'{SEQUENCES_DIR}/{seq}/image_2'
    cam_dest = f'{img_dir}/cam.txt'
    if os.path.exists(cam_src) and os.path.exists(img_dir):
        shutil.copy(cam_src, cam_dest)
        print(f'✅ sequences/{seq}/image_2/cam.txt installed')
    else:
        print(f'⚠️  Missing: {cam_src} or {img_dir}')

# Count images in each sequence
import glob
for seq in ['09', '10']:
    imgs = glob.glob(f'{SEQUENCES_DIR}/{seq}/image_2/*.png')
    print(f'Seq {seq}: {len(imgs)} images')

In [ ]:
# ── CELL 8: Run Offline VO Evaluation (Seq 09 & 10) ──────────────────────────
import os
os.makedirs('vo_results', exist_ok=True)

POSE_NET   = 'checkpoints/exp_pose112_model_best.pth.tar'
DATASET    = 'data/kitti_odom/sequences/'
OUTPUT_DIR = 'vo_results/'

for seq in ['09', '10']:
    print(f'\n=== Running VO for sequence {seq} ===')
    !CUDA_VISIBLE_DEVICES=0 python test_vo.py \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --dataset-dir {DATASET} \
        --output-dir {OUTPUT_DIR}

print('\n=== VO Results Files ===')
!ls -lh vo_results/*.txt 2>/dev/null || echo 'No results yet'

In [ ]:
# ── CELL 9: Evaluate VO Accuracy ─────────────────────────────────────────────
print('=== Evaluating VO Accuracy (7-DoF alignment) ===')
!python ./kitti_eval/eval_odom.py --result=vo_results/ --align='7dof'

In [ ]:
# ── CELL 10: Run Online VO Evaluation (Seq 09) ───────────────────────────────
import os
os.makedirs('vo_results_online', exist_ok=True)

POSE_NET   = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET   = 'checkpoints/dispnet112_model_best.pth.tar'
DATASET    = 'data/kitti_odom/sequences/'
OUTPUT_DIR = 'vo_results_online/'

print('=== Running Online VO for sequence 09 ===')
!CUDA_VISIBLE_DEVICES=0 python test_vo_online.py \
    -p 1 -s 0.1 -c 0.5 \
    --img-height 256 --img-width 832 \
    --sequence 09 \
    --pretrained-posenet {POSE_NET} \
    --pretrained-disp    {DISP_NET} \
    --dataset-dir {DATASET} \
    --output-dir  {OUTPUT_DIR} \
    --epochs 2 --lr 1e-4 \
    --sequence-length 3 \
    --resnet-layers 50 \
    --select-best 1 \
    --with-mask 1 \
    --with-auto-mask 1 \
    --padding-mode border

In [ ]:
# ── CELL 11: Evaluate Online VO ───────────────────────────────────────────────
print('=== Evaluating Online VO Accuracy ===')
!python ./kitti_eval/eval_odom.py --result=vo_results_online/ --align='7dof'

In [ ]:
# ── CELL 12 (Optional): Download Results ────────────────────────────────────
from google.colab import files
import zipfile, os

# Zip results
with zipfile.ZipFile('/content/sgvo_results.zip', 'w') as z:
    for d in ['vo_results', 'vo_results_online']:
        for f in os.listdir(d):
            z.write(os.path.join(d, f))

print('Downloading results zip...')
files.download('/content/sgvo_results.zip')